# Vígil.ia — 11n @640: base 12,5k + fine-tune nas fotos reais (v2 limpo)

Pipeline em **2 estágios @640**, com todas as correções recentes:

```
COCO -> ESTÁGIO 1: base 12,5k (Roboflow, pseudo-rótulo Otsu)
     -> ESTÁGIO 2: fine-tune fotos reais (Soja total/Lotes + Soja pra completar)
                   + vídeos de DEFEITO (quando existirem; intact fica FORA)
```

**Correções herdadas da rodada anterior:**
| # | Correção |
|---|---|
| 1 | Otsu v2 no vídeo: dedupe anti-sombra, forma ≤2.2, preenchimento ≥0.5, margem de borda |
| 2 | `make_scene` com escala proporcional ao canvas |
| 3 | Vídeos de **intacto fora do treino** → avaliação limpa |

**⚠️ Registro honesto:** este caminho (base 12,5k) já foi testado no 11n e perdeu
feio do direto (18,7% vs 79,5% no vídeo — o estágio base "cozinha" o modelo no fundo
preto). Esta rodada refaz o teste **limpo** (sem contaminação, correções aplicadas);
o placar final compara com o campeão direto e decide. Se reproduzir a derrota,
encerramos o assunto base-12,5k com evidência dupla.

**Economia:** estágio 1 é cacheado no Drive (`soja_yolo11n_12k_stage1.pt` — se você
já rodou o `treino_yolo11s_e_11n`, ele é reaproveitado e pula ~1h).

## 0. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU!'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Caminhos + config

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

CHAMPION_PT = '/content/drive/MyDrive/soja_yolo11n_baseline.pt'   # referência a bater
assert os.path.exists(CHAMPION_PT), f'campeão não encontrado: {CHAMPION_PT}'

# fotos reais -> estágio 2 (fine-tune)
REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]
# 12,5k Roboflow -> estágio 1 (base)
CLS_BASE_CANDS = [
    '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder',
    '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)',
]
CLS_BASE = next((p for p in CLS_BASE_CANDS if os.path.isdir(p)), None)
assert CLS_BASE, 'dataset 12,5k não encontrado:\n  ' + '\n  '.join(CLS_BASE_CANDS)

VAL_ROOT = '/content/drive/MyDrive/Vídeos para treino/Treino'
assert os.path.isdir(VAL_ROOT), f'VAL_ROOT não existe: {VAL_ROOT}'

SIZE = 640
FRAME_STRIDE = 5
EXCLUIR_DO_TREINO = {'intact'}   # intact NUNCA treina -> avaliação limpa
print('base 12,5k:', CLS_BASE)
print('fine-tune :', REAL_SRCS)
print('fora do treino:', EXCLUIR_DO_TREINO)

## 2. Funções de dataset + `make_scene` proporcional (correção #2)

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml
import numpy as np

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RNG = np.random.default_rng(42)

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def motion_blur(img, rng=RNG):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    x, y, bw, bh = cv2.boundingRect(c)
    return img[y:y + bh, x:x + bw], mask[y:y + bh, x:x + bw]

def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

def balance_train(items):
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('balanceado (train):', {NAMES[c]: sum(1 for it in out if it[1] == c) for c in sorted(by)})
    return out + rest

def build_v3(items, out_dir, n_synth=600, blur_frac=0.4):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1; continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(line)
        kept += 1
        if sp == 'train':
            cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
            kept += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | recortes: {len(cutouts)}')
    assert cutouts, 'Nenhum recorte extraído!'
    synth = 0
    for j in range(n_synth):
        if j % 100 == 0:
            print(f'  cenas {j}/{n_synth}…', flush=True)
        canvas, boxes = make_scene(cutouts)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        stem = f'synth_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in boxes))
        synth += 1
    print(f'cenas sintéticas: {synth}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}

def collect_base(base_dir):
    """Acha train/valid/test em qualquer profundidade dentro do dataset 12,5k."""
    items = []
    for root, dirs, _ in os.walk(base_dir):
        for d in list(dirs):
            sp = SPLIT_MAP.get(d.lower())
            if sp is None:
                continue
            split_dir = os.path.join(root, d)
            for folder in sorted(os.listdir(split_dir)):
                cls = class_of(folder)
                if cls is None:
                    continue
                for p_ in glob.glob(os.path.join(split_dir, folder, '*')):
                    if p_.lower().endswith(IMG_EXT):
                        items.append((p_, cls, sp))
            dirs.remove(d)
    from collections import Counter
    print('coletado (base 12,5k):', dict(Counter(sp for _, _, sp in items)))
    return items

def build_base(items, out_dir):
    """Dataset base de detecção: pseudo-rótulo Otsu (1 grão/img, fundo preto),
    sem balanceamento/blur/sintético — idêntico ao estágio base do RT-DETR."""
    assert items, 'Nenhuma imagem do 12,5k coletada!'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 1000 == 0:
            print(f'  base {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = otsu_box(img)
        if box is None:
            skipped += 1; continue
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(
            f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        kept += 1
    print(f'base: kept={kept} skipped={skipped}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

# --- correção #2: escala do grão proporcional ao canvas (identidade em 640,
#     mas mantém o pipeline consistente com o notebook 1280) ---
def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    lo, hi = int(60 * size / 640), int(150 * size / 640)
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(lo, hi)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

print('funções prontas (make_scene proporcional)')

## 3. Constrói os 2 datasets
- **BASE 12,5k** (estágio 1): pseudo-rótulo Otsu, 1 grão/img, sem balanceamento/sintético.
- **v3 fotos reais** (estágio 2): Lotes + pra completar, balanceado + blur + cenas sintéticas.

In [ ]:
BASE_YAML = '/content/soja_det_base/data.yaml'
if not os.path.exists(BASE_YAML):
    print('construindo base 12,5k (~10 min)…')
    BASE_YAML = build_base(collect_base(CLS_BASE), '/content/soja_det_base')
print('base 12,5k:', BASE_YAML)

V3_YAML = '/content/soja_det_v3/data.yaml'
if not os.path.exists(V3_YAML):
    print('construindo v3 fotos reais (~10-15 min)…')
    V3_YAML = build_v3(collect_real(REAL_SRCS), '/content/soja_det_v3')
print('v3 fotos reais:', V3_YAML)

## 4. Otsu v2 (correção #1) + vídeos de DEFEITO no fine-tune (correção #3)
Intact é pulado (vira avaliação). Hoje: nenhum vídeo entra; quando você gravar
defeitos, eles entram sozinhos no estágio 2.

In [ ]:
VIDEO_EXT = ('.mp4', '.mov', '.avi', '.mkv')
ROI_BOTTOM = 0.15
MIN_SAT    = 40

def video_frames(path, stride=FRAME_STRIDE):
    cap = cv2.VideoCapture(path)
    k = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if k % stride == 0:
            yield k, frame[:int(frame.shape[0] * (1 - ROI_BOTTOM))]
        k += 1
    cap.release()

def _dedupe(boxes):
    out = []
    boxes = sorted(boxes, key=lambda b: -(b[2] * b[3]))
    for b in boxes:
        bx1, by1, bx2, by2 = b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
        dup = False
        for k in out:
            kx1, ky1, kx2, ky2 = k[0]-k[2]/2, k[1]-k[3]/2, k[0]+k[2]/2, k[1]+k[3]/2
            iw = max(0, min(bx2, kx2) - max(bx1, kx1))
            ih = max(0, min(by2, ky2) - max(by1, ky1))
            if iw * ih > 0.4 * (b[2] * b[3]):
                dup = True; break
        if not dup:
            out.append(b)
    return out

def multi_boxes(img, min_frac=0.0015, max_frac=0.05, edge=0.02):
    h, w = img.shape[:2]
    S = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[:, :, 1]
    blur = cv2.GaussianBlur(S, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in cnts:
        a = cv2.contourArea(c)
        if not (min_frac * h * w <= a <= max_frac * h * w):
            continue
        x, y, bw, bh = cv2.boundingRect(c)
        if bw / max(bh, 1) > 2.2 or bh / max(bw, 1) > 2.2:
            continue
        if a < 0.5 * bw * bh:
            continue
        if S[y:y+bh, x:x+bw].mean() < MIN_SAT:
            continue
        cx, cy = (x + bw/2) / w, (y + bh/2) / h
        if not (edge < cx < 1-edge and edge < cy < 1-edge):
            continue
        pad = int(0.04 * min(bw, bh)) + 2
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
        boxes.append((((x1+x2)/2)/w, ((y1+y2)/2)/h, (x2-x1)/w, (y2-y1)/h))
    return _dedupe(boxes)

VID = '/content/soja_video_640'
import shutil as _sh
_sh.rmtree(VID, ignore_errors=True)
for sp in ('train', 'val'):
    os.makedirs(f'{VID}/images/{sp}', exist_ok=True)
    os.makedirs(f'{VID}/labels/{sp}', exist_ok=True)

stats = {}
for entry in sorted(os.listdir(VAL_ROOT)):
    sub = os.path.join(VAL_ROOT, entry)
    if not os.path.isdir(sub):
        continue
    cls = class_of(entry)
    if cls is None:
        continue
    if NAMES[cls] in EXCLUIR_DO_TREINO:
        print(f'{entry} -> {NAMES[cls]}: EXCLUÍDO do treino (avaliação limpa)')
        continue
    vids = [os.path.join(sub, f) for f in sorted(os.listdir(sub))
            if f.lower().endswith(VIDEO_EXT)]
    if not vids:
        continue
    val_video = vids[-1] if len(vids) >= 2 else None
    wrote = {'train': 0, 'val': 0}
    for vi, vp in enumerate(vids):
        frames = list(video_frames(vp))
        if val_video is None:
            cut = int(0.85 * len(frames))
            split_of = lambda i: ('train' if i < cut else
                                  None if i < cut + 150 // FRAME_STRIDE else 'val')
        else:
            split_of = lambda i: 'val' if vp == val_video else 'train'
        for i, (k, frame) in enumerate(frames):
            sp = split_of(i)
            if sp is None:
                continue
            boxes = multi_boxes(frame)
            if not (3 <= len(boxes) <= 80):
                continue
            lb, s, left, top = letterbox640(frame, size=SIZE)
            H0, W0 = frame.shape[:2]
            lines = []
            for cx, cy, ww, hh in boxes:
                cx2 = (cx * W0 * s + left) / SIZE
                cy2 = (cy * H0 * s + top) / SIZE
                lines.append(f'{cls} {cx2:.6f} {cy2:.6f} {ww*W0*s/SIZE:.6f} {hh*H0*s/SIZE:.6f}')
            stem = f'{NAMES[cls]}_{vi}_{k:05d}'
            cv2.imwrite(f'{VID}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{VID}/labels/{sp}/{stem}.txt', 'w').write('\n'.join(lines))
            wrote[sp] += 1
    stats[NAMES[cls]] = wrote
    print(f'{entry} -> {NAMES[cls]}: {wrote}')

HAS_VIDEO_DATA = any(v['train'] for v in stats.values())
print('\nvídeo no fine-tune:', stats if HAS_VIDEO_DATA else
      'NENHUM (só intact existe e está excluído) — estágio 2 será só fotos reais')

## 5. Treino em 2 estágios
**Estágio 1** (12,5k, cacheado no Drive) → **Estágio 2** (fotos reais + defeitos se houver).
Salva `soja_yolo11n_base12k_v2.pt`.

In [ ]:
import yaml
from ultralytics import YOLO
import shutil

COMMON = dict(
    imgsz=SIZE, device=0, seed=42, optimizer='AdamW',
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_640', exist_ok=True,
)
BATCH = 32

# ---- estágio 1: base 12,5k (cache no Drive; reaproveita o do treino_yolo11s_e_11n) ----
STAGE1 = '/content/drive/MyDrive/soja_yolo11n_12k_stage1.pt'
if os.path.exists(STAGE1):
    print('estágio 1 já no Drive — reaproveitando:', STAGE1)
else:
    print('ESTÁGIO 1: base 12,5k (~1h na L4)')
    mb = YOLO('yolo11n.pt')
    mb.train(name='11n_base12k', data=BASE_YAML, batch=BATCH, epochs=50,
             lr0=0.001, patience=20, close_mosaic=10, **COMMON)
    shutil.copy(str(mb.trainer.best), STAGE1)
    print('estágio 1 salvo:', STAGE1)

# ---- estágio 2: fine-tune fotos reais (+ vídeos de defeito se houver) ----
MIX_YAML = '/content/soja_mix_640.yaml'
train_dirs = ['/content/soja_det_v3/images/train']
if HAS_VIDEO_DATA:
    train_dirs.append(f'{VID}/images/train')
yaml.safe_dump({'train': train_dirs,
                'val': '/content/soja_det_v3/images/val',
                'names': {i: n for i, n in enumerate(NAMES)}},
               open(MIX_YAML, 'w'), sort_keys=False, allow_unicode=True)
print('fine-tune:', train_dirs)

DST = '/content/drive/MyDrive/soja_yolo11n_base12k_v2.pt'
if os.path.exists(DST):
    print('já treinado:', DST)
else:
    m = YOLO(STAGE1)
    m.train(name='11n_base12k_v2', data=MIX_YAML, batch=BATCH, epochs=60,
            lr0=0.001, patience=20, close_mosaic=8, **COMMON)
    shutil.copy(str(m.trainer.best), DST)
    print('salvo:', DST)

## 6. Placar — base12k_v2 vs campeão direto (a decisão)
Vídeos de intacto = avaliação **limpa** (nenhum dos dois treinou neles) + mAP por
classe no v3 val. A bater: campeão direto = **79,5%** (e o base12k antigo fez 18,7%).

In [ ]:
import time
from collections import Counter
from ultralytics import YOLO

DST = '/content/drive/MyDrive/soja_yolo11n_base12k_v2.pt'
CASOS = [('campeão direto', CHAMPION_PT), ('base12k_v2', DST)]

by_class = {}
for entry in sorted(os.listdir(VAL_ROOT)):
    sub = os.path.join(VAL_ROOT, entry)
    if not os.path.isdir(sub):
        continue
    c = class_of(entry)
    if c is None:
        continue
    vids = [os.path.join(sub, f) for f in sorted(os.listdir(sub))
            if f.lower().endswith(VIDEO_EXT)]
    if vids:
        by_class[NAMES[c]] = vids
print('classes com vídeo (avaliação):', {k: len(v) for k, v in by_class.items()})

def video_eval(pt, imgsz=SIZE, vid_stride=5):
    model = YOLO(pt)
    tot, cert = Counter(), Counter()
    for true_cls, vids in by_class.items():
        for vid in vids:
            for r in model.predict(source=vid, imgsz=imgsz, conf=0.35, iou=0.5,
                                   agnostic_nms=True, vid_stride=vid_stride,
                                   stream=True, verbose=False):
                for c in r.boxes.cls.int().tolist():
                    tot[true_cls] += 1
                    if NAMES[c] == true_cls:
                        cert[true_cls] += 1
    return tot, cert

print('\n=== eixo 1: vídeos por classe (limpo p/ ambos) ===')
for tag, pt in CASOS:
    if not os.path.exists(pt):
        print(f'{tag}: ausente — pulado'); continue
    tot, cert = video_eval(pt)
    n, ok = sum(tot.values()), sum(cert.values())
    print(f'  {tag:16s}: {ok}/{n} = {100*ok/max(n,1):.1f}%  ({n} det)')

print('\n=== eixo 2: mAP no v3 val (por classe) ===')
for tag, pt in CASOS:
    if not os.path.exists(pt):
        continue
    r = YOLO(pt).val(data=V3_YAML, split='val', imgsz=SIZE, device=0, verbose=False)
    per = {NAMES[int(i)]: f'{ap:.2f}' for i, ap in
           zip(r.box.ap_class_index, r.box.maps[r.box.ap_class_index])}
    print(f'  {tag:16s}: mAP50={r.box.map50:.3f}  por classe: {per}')

print('\nDecisão: base12k_v2 só substitui o campeão se vencer nos DOIS eixos.')
print('Se repetir a derrota do base12k antigo (18,7%), o assunto encerra com')
print('evidência dupla: pro nano, direto > base 12,5k.')

## Depois do placar
- Venceu? `soja_yolo11n_base12k_v2.pt` vira o novo campeão — atualize `CHAMPION_PT`
  nos outros notebooks e rode o bloco do `teste_soja` pra confirmar no vídeo misto.
- Perdeu? O campeão direto segue absoluto e o estágio 12,5k sai da receita do nano
  em definitivo. Nada foi perdido: o estágio 1 fica cacheado no Drive.
- Em ambos os casos, o desbloqueio real continua sendo os vídeos de defeito — eles
  entram sozinhos no estágio 2 deste notebook quando existirem.